# Проверяем ряды на стационарность

Сначала проверим новые ряды для Российского домена, потом остальные

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

In [2]:
dataset = pd.read_excel("../russian_data/processed/DATASET-extended.xlsx")

In [5]:
dataset

,date,ret_t,rpo,rea_t,delta_world,delta_non_rus,BRENT,cpi_us,log_real_brent
0,2000-01-01,20.508306,2.703907,-10.113044,0.274989,0.187119,25.5900,169.300,-1.889471
1,2000-02-01,-0.895221,2.779550,-8.766956,1.106906,1.115102,25.3800,170.000,-1.901837
2,2000-03-01,27.505884,2.784939,5.909950,0.274396,0.236799,27.7048,171.000,-1.820058
3,2000-04-01,-5.413489,2.654156,8.995164,0.765926,0.817340,27.4700,170.900,-1.827984
4,2000-05-01,-12.123873,2.733806,5.009274,0.599138,0.575270,22.5400,171.200,-2.027541
...,...,...,...,...,...,...,...,...,...
175,2014-08-01,1.029960,3.670241,-52.811092,0.425677,0.410585,106.9800,237.460,-0.797357
176,2014-09-01,0.493853,3.627996,-34.578872,0.933678,1.036886,101.9200,237.477,-0.845883
177,2014-10-01,4.697298,3.551121,-36.350448,1.234880,1.275204,97.3400,237.430,-0.891663
178,2014-11-01,2.173902,3.445831,-16.933751,-0.295374,-0.334066,87.2700,236.983,-0.998982


In [6]:
dataset['delta_rea'] = dataset['rea_t'].diff(1)

In [8]:
dataset['delta_rpo'] = dataset['rpo'].diff(1)

In [14]:
dataset_cur = pd.read_excel("../russian_data/processed/DATASET-rub.xlsx")

In [16]:
dataset_cur.columns

Index(['date', 'delta_non_rus', 'ret_t', 'rpo', 'IGREA', 'rub_diff'], dtype='str')

## Сетка графиков

In [13]:
from plotly.subplots import make_subplots

def add_series(fig, row, col, x, y, title,
               fill=True,
               color="#1f77b4"):

    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="lines",
            line=dict(color=color, width=2),
            fill="tozeroy" if fill else None,
            fillcolor="rgba(31,119,180,0.15)",
            showlegend=False
        ),
        row=row,
        col=col
    )

    fig.update_xaxes(
        tickformat="%Y",
        gridcolor="lightgray",
        showgrid=True,
        showline=True,
        linewidth=1,
        linecolor="black",
        row=row,
        col=col
    )

    fig.update_yaxes(
        gridcolor="lightgray",
        showgrid=True,
        showline=True,
        linewidth=1,
        linecolor="black",
        row=row,
        col=col
    )

# ==========================================================
# СЕТКА 1
# Ряды, которые сначала анализировались в уровнях, затем в разностях
# ==========================================================

fig1 = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "IGREA (уровни)",
        "Первая разность IGREA",
        "RPO (уровни)",
        "Первая разность RPO"
    ]
)

add_series(
    fig1, 1, 1,
    dataset["date"],
    dataset["rea_t"],          # если у вас другое имя столбца IGREA — замените
    "IGREA",
    fill=False
)

add_series(
    fig1, 1, 2,
    dataset["date"],
    dataset["delta_rea"],      # delta_igrea
    "Первая разность IGREA"
)

add_series(
    fig1, 2, 1,
    dataset["date"],
    dataset["rpo"],
    "RPO",
    fill=False
)

add_series(
    fig1, 2, 2,
    dataset["date"],
    dataset["delta_rpo"],
    "Первая разность RPO"
)

fig1.update_layout(
    title="Переменные и их первые разности",
    template="plotly_white",
    height=600,
    width=1200,
    hovermode="x unified"
)

fig1.show()

pio.write_image(
    fig1,
    "../russian_paper_results/graphs/variables_and_diffs.png",
    scale=2
)

# ==========================================================
# СЕТКА 2
# Ряды, которые оказались стационарными сразу
# ==========================================================

fig2 = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "BRENT",
        "Первая разность World Production",
        "Первая разность Non-Russia Production",
        "RET"
    ]
)

add_series(
    fig2, 1, 1,
    dataset["date"],
    dataset["log_real_brent"],   # BRENT
    "BRENT",
    fill=False
)

add_series(
    fig2, 1, 2,
    dataset["date"],
    dataset["delta_world"],
    "Первая разность World"
)

add_series(
    fig2, 2, 1,
    dataset["date"],
    dataset["delta_non_rus"],
    "Первая разность Non-Russia"
)

add_series(
    fig2, 2, 2,
    dataset["date"],
    dataset["ret_t"],            # RET
    "RET"
)

fig2.update_layout(
    title="Стационарные и переменные + BRENT",
    template="plotly_white",
    height=600,
    width=1200,
    hovermode="x unified"
)

fig2.show()

pio.write_image(
    fig2,
    "../russian_paper_results/graphs/stationary_variables.png",
    scale=2
)

In [19]:
dataset_cur.tail()

,date,delta_non_rus,ret_t,rpo,IGREA,rub_diff
79,2021-08-01,-0.567196,3.522422,3.182289,85.238194,1.4091
80,2021-09-01,0.157638,4.431849,3.230383,99.163111,-0.3245
81,2021-10-01,1.772758,0.528499,3.312722,110.066030,-0.7019
82,2021-11-01,1.170472,-7.556297,3.312976,54.347524,-1.3990
83,2021-12-01,-0.671492,-3.645191,3.190105,55.679372,1.0944


In [18]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio

# ==========================================================
# СЕТКА ДЛЯ ДАННЫХ ПОСЛЕ 2015 ГОДА
# ==========================================================

fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=[
        "Первая разность Non-Russia Production",
        "Первая разность RUB",
        "RET",
        "RPO",
        "IGREA",
        ""
    ]
)

# Первая разность Non-Russia Production
add_series(
    fig, 1, 1,
    dataset_cur["date"],
    dataset_cur["delta_non_rus"],
    "Первая разность Non-Russia"
)

# Первая разность RUB
add_series(
    fig, 1, 2,
    dataset_cur["date"],
    dataset_cur["rub_diff"],
    "Первая разность RUB"
)

# RET
add_series(
    fig, 2, 1,
    dataset_cur["date"],
    dataset_cur["ret_t"],
    "RET"
)

# RPO
add_series(
    fig, 2, 2,
    dataset_cur["date"],
    dataset_cur["rpo"],
    "RPO",
    fill=False
)

# IGREA
add_series(
    fig, 3, 1,
    dataset_cur["date"],
    dataset_cur["IGREA"],
    "IGREA",
    fill=False
)

fig.update_layout(
    title="Графики переменных после 2015 года",
    template="plotly_white",
    height=800,
    width=1200,
    hovermode="x unified"
)

fig.show()

pio.write_image(
    fig,
    "../russian_paper_results/graphs/post_2015_variables.png",
    scale=2
)

## Мировая добыча без учета РФ

In [7]:
delta_non_rus = dataset[['date', 'delta_non_rus']]

In [21]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=delta_non_rus['date'],
    y=delta_non_rus['delta_non_rus'],
    mode='lines',
    name='Процентное изменение мировой добычи нефти без учета России',
    line=dict(color='#1f77b4', width=2),
    fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.15)'
))
fig.update_layout(
    title=dict(
        text='<b>Процентное изменение мировой добычи нефти без учета России</b><br>' +
             '<span style="font-size:12px; color:gray;">Процентное изменение (log разности, х100)</span>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    xaxis=dict(
        title=dict(text='Месяц', font=dict(size=12)),
        tickformat='%Y',
        gridcolor='lightgray',
        showgrid=True,
        zeroline=False,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    yaxis=dict(
        title=dict(text='Процентное изменение (%)', font=dict(size=12)),
        gridcolor='lightgray',
        showgrid=True,
        zeroline=True,
        zerolinecolor='black',
        zerolinewidth=1,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=500,
    width=900,
    hovermode='x unified'
)

fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)

pio.write_image(fig, '../russian_paper_results/graphs/delta_non_rus.png', scale=2)
fig.show()

Используем значит для него спецификацию без тренда и дрифта

## Мировая добыча

In [22]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dataset['date'],
    y=dataset['delta_world'],
    mode='lines',
    name='Процентное изменение мировой добычи нефти',
    line=dict(color='#1f77b4', width=2),
    fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.15)'
))
fig.update_layout(
    title=dict(
        text='<b>Процентное изменение мировой добычи нефти</b><br>' +
             '<span style="font-size:12px; color:gray;">Процентное изменение (log разности, х100)</span>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    xaxis=dict(
        title=dict(text='Месяц', font=dict(size=12)),
        tickformat='%Y',
        gridcolor='lightgray',
        showgrid=True,
        zeroline=False,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    yaxis=dict(
        title=dict(text='Процентное изменение (%)', font=dict(size=12)),
        gridcolor='lightgray',
        showgrid=True,
        zeroline=True,
        zerolinecolor='black',
        zerolinewidth=1,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=500,
    width=900,
    hovermode='x unified'
)

fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)

pio.write_image(fig, '../russian_paper_results/graphs/delta_world.png', scale=2)
fig.show()

Без детерминированных компонент

## Доходность российских акций

In [23]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dataset['date'],
    y=dataset['ret_t'],
    mode='lines',
    name='Доходность российских акций',
    line=dict(color='#1f77b4', width=2),
    fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.15)'
))
fig.update_layout(
    title=dict(
        text='<b>Доходность российских акций</b><br>' +
             '<span style="font-size:12px; color:gray;">Процентное изменение (log_return минус инфляция по ИПЦ)</span>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    xaxis=dict(
        title=dict(text='Месяц', font=dict(size=12)),
        tickformat='%Y',
        gridcolor='lightgray',
        showgrid=True,
        zeroline=False,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    yaxis=dict(
        title=dict(text='Процентное изменение (%)', font=dict(size=12)),
        gridcolor='lightgray',
        showgrid=True,
        zeroline=True,
        zerolinecolor='black',
        zerolinewidth=1,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=500,
    width=900,
    hovermode='x unified'
)

fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)

pio.write_image(fig, '../russian_paper_results/graphs/ret_ru.png', scale=2)
fig.show()

Без дерерминированных компонент

## Индекс экономической активности

In [25]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dataset['date'],
    y=dataset['rea_t'],
    mode='lines',
    name='Индекс экономической активности',
    line=dict(color='#1f77b4', width=2),
    fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.15)'
))
fig.update_layout(
    title=dict(
        text='<b>Индекс экономической активности Killian</b><br>' +
             '<span style="font-size:12px; color:gray;">Логарифмы уровней</span>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    xaxis=dict(
        title=dict(text='Месяц', font=dict(size=12)),
        tickformat='%Y',
        gridcolor='lightgray',
        showgrid=True,
        zeroline=False,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    yaxis=dict(
        title=dict(text='Логарифм', font=dict(size=12)),
        gridcolor='lightgray',
        showgrid=True,
        zeroline=True,
        zerolinecolor='black',
        zerolinewidth=1,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=500,
    width=900,
    hovermode='x unified'
)

fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)

pio.write_image(fig, '../russian_paper_results/graphs/igrea.png', scale=2)
fig.show()

используем без детерминированных компонент

## Мировая цена нефти (rpo)

In [27]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dataset['date'],
    y=dataset['rpo'],
    mode='lines',
    name='Мировая цена нефти',
    line=dict(color='#1f77b4', width=2),
    #fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.15)'
))
fig.update_layout(
    title=dict(
        text='<b>Мировая цена нефти</b><br>' +
             '<span style="font-size:12px; color:gray;">Логарифмы уровней</span>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    xaxis=dict(
        title=dict(text='Месяц', font=dict(size=12)),
        tickformat='%Y',
        gridcolor='lightgray',
        showgrid=True,
        zeroline=False,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    yaxis=dict(
        title=dict(text='Логарифм', font=dict(size=12)),
        gridcolor='lightgray',
        showgrid=True,
        zeroline=True,
        zerolinecolor='black',
        zerolinewidth=1,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=500,
    width=900,
    hovermode='x unified'
)

fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)

pio.write_image(fig, '../russian_paper_results/graphs/rpo.png', scale=2)
fig.show()

Есть небольшой тренд и дрифт

## Второй вариант цены на нефть

In [30]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dataset['date'],
    y=dataset['log_real_brent'],
    mode='lines',
    name='Мировая цена нефти (BRENT)',
    line=dict(color='#1f77b4', width=2),
    # fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.15)'
))
fig.update_layout(
    title=dict(
        text='<b>Мировая цена нефти (BRENT)</b><br>' +
             '<span style="font-size:12px; color:gray;">Логарифмы уровней</span>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    xaxis=dict(
        title=dict(text='Месяц', font=dict(size=12)),
        tickformat='%Y',
        gridcolor='lightgray',
        showgrid=True,
        zeroline=False,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    yaxis=dict(
        title=dict(text='Логарифм', font=dict(size=12)),
        gridcolor='lightgray',
        showgrid=True,
        zeroline=True,
        zerolinecolor='black',
        zerolinewidth=1,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=500,
    width=900,
    hovermode='x unified'
)

# fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)

pio.write_image(fig, '../russian_paper_results/graphs/log_real_brent.png', scale=2)
fig.show()

Констранта и тренд

# Анализ показал, что rpo и rea в моделях не стационарны

Возьмем разности

In [32]:
dataset['delta_rpo'] = dataset['rpo'].diff(1)

In [34]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dataset['date'],
    y=dataset['delta_rpo'],
    mode='lines',
    name='Мировая цена нефти в разностях',
    line=dict(color='#1f77b4', width=2),
    fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.15)'
))
fig.update_layout(
    title=dict(
        text='<b>Мировая цена нефти (BRENT)</b><br>' +
             '<span style="font-size:12px; color:gray;">Разность логарифмов</span>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    xaxis=dict(
        title=dict(text='Месяц', font=dict(size=12)),
        tickformat='%Y',
        gridcolor='lightgray',
        showgrid=True,
        zeroline=False,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    yaxis=dict(
        title=dict(text='Первая разность', font=dict(size=12)),
        gridcolor='lightgray',
        showgrid=True,
        zeroline=True,
        zerolinecolor='black',
        zerolinewidth=1,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=500,
    width=900,
    hovermode='x unified'
)

# fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)

pio.write_image(fig, '../russian_paper_results/graphs/delta_rpo.png', scale=2)
fig.show()

In [36]:
dataset['delta_rea'] = dataset['rea_t'].diff(1)

In [37]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dataset['date'],
    y=dataset['delta_rea'],
    mode='lines',
    name='Индекс мировой деловой активности в разностях',
    line=dict(color='#1f77b4', width=2),
    fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.15)'
))
fig.update_layout(
    title=dict(
        text='<b>Индекс мировой деловой активности в разностях</b><br>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    xaxis=dict(
        title=dict(text='Месяц', font=dict(size=12)),
        tickformat='%Y',
        gridcolor='lightgray',
        showgrid=True,
        zeroline=False,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    yaxis=dict(
        title=dict(text='Первая разность', font=dict(size=12)),
        gridcolor='lightgray',
        showgrid=True,
        zeroline=True,
        zerolinecolor='black',
        zerolinewidth=1,
        showline=True,
        linewidth=1,
        linecolor='black',
        ticks='outside'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=500,
    width=900,
    hovermode='x unified'
)

# fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)

pio.write_image(fig, '../russian_paper_results/graphs/delta_igrea.png', scale=2)
fig.show()

In [43]:
dataset.to_excel("../russian_data/processed/DATASET-stationary.xlsx", index=False)

In [42]:
dataset = dataset.dropna()